# Understand Embeddings 

In [ ]:
!pip install python-dotenv num2words pandas numpy tiktoken

In [2]:
import os
import re
import requests
import sys
from num2words import num2words
import os
import pandas as pd
import numpy as np
import tiktoken
from openai import AzureOpenAI

In [4]:
# Import data
df=pd.read_csv(os.path.join(os.getcwd(),'movies.csv'))
df

,Unnamed: 0,Title,Movie Info,Year,Distributor,Budget (in $),Domestic Opening (in $),Domestic Sales (in $),International Sales (in $),World Wide Sales (in $),Release Date,Genre,Running Time,License
0,0,Avatar,A paraplegic Marine dispatched to the moon Pan...,2009,Twentieth Century Fox,237000000,77025481,785221649,2138484377,2923706026,16-Dec-09,"['Action', 'Adventure', 'Fantasy', 'Sci-Fi']",2 hr 42 min,PG-13
1,1,Avengers: Endgame,After the devastating events of Avengers: Infi...,2019,Walt Disney Studios Motion Pictures,356000000,357115007,858373000,1941066100,2799439100,24-Apr-19,"['Action', 'Adventure', 'Drama', 'Sci-Fi']",3 hr 1 min,PG-13
2,2,Avatar: The Way of Water,Jake Sully lives with his newfound family form...,2022,20th Century Studios,December 14 2022 (EMEA APAC),134100226,684075767,1636174514,2320250281,24-Apr-19,"['Action', 'Adventure', 'Drama', 'Sci-Fi']",3 hr 1 min,PG-13
3,3,Titanic,A seventeen-year-old aristocrat falls in love ...,1997,Paramount Pictures,200000000,28638131,674292608,1590450697,2264743305,19-Dec-97,"['Drama', 'Romance']",3 hr 14 min,PG-13
4,4,Star Wars: Episode VII - The Force Awakens,"As a new threat to the galaxy rises, Rey, a de...",2015,Walt Disney Studios Motion Pictures,245000000,247966675,936662225,1134647993,2071310218,16-Dec-15,"['Action', 'Adventure', 'Sci-Fi']",2 hr 18 min,PG-13
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,995,Sweet Home Alabama,A young woman who has reinvented herself as a ...,2002,Walt Disney Studios Motion Pictures,30000000,35648740,127223418,53399006,180622424,27-Sep-02,"['Comedy', 'Romance']",1 hr 48 min,PG-13
996,996,Daddy's Home 2,Having finally gotten used to each other's exi...,2017,Paramount Pictures,69000000,29651193,104029443,76584381,180613824,9-Nov-17,['Comedy'],1 hr 40 min,PG-13
997,997,Hacksaw Ridge,World War II American Army Medic Desmond T. Do...,2016,Lionsgate,40000000,15190758,67209615,113354021,180563636,3-Nov-16,"['Biography', 'Drama', 'History', 'War']",2 hr 19 min,R
998,998,Deja Vu,"After a ferry is bombed in New Orleans, an A.T...",2006,Walt Disney Studios Motion Pictures,75000000,20574802,64038616,116518934,180557550,22-Nov-06,"['Action', 'Crime', 'Sci-Fi', 'Thriller']",2 hr 6 min,PG-13


In [6]:
df_info = df[["Title", "Movie Info", "Genre"]]
df_info.head()

,Title,Movie Info,Genre
0,Avatar,A paraplegic Marine dispatched to the moon Pan...,"['Action', 'Adventure', 'Fantasy', 'Sci-Fi']"
1,Avengers: Endgame,After the devastating events of Avengers: Infi...,"['Action', 'Adventure', 'Drama', 'Sci-Fi']"
2,Avatar: The Way of Water,Jake Sully lives with his newfound family form...,"['Action', 'Adventure', 'Drama', 'Sci-Fi']"
3,Titanic,A seventeen-year-old aristocrat falls in love ...,"['Drama', 'Romance']"
4,Star Wars: Episode VII - The Force Awakens,"As a new threat to the galaxy rises, Rey, a de...","['Action', 'Adventure', 'Sci-Fi']"


In [7]:
# Import the tokenizer from the tiktoken library and get the encoding for "cl100k_base"
tokenizer = tiktoken.get_encoding("cl100k_base")

# Apply the tokenizer to the "Movie Info" column in the dataframe df_info
# For each entry in "Movie Info", encode the text and calculate the number of tokens
# Store the result in a new column "num_tokens"
df_info['num_tokens'] = df_info["Movie Info"].apply(lambda x: len(tokenizer.encode(x)))
df_info

C:\Users\hoangma\AppData\Local\Temp\ipykernel_13316\1769520982.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_info['num_tokens'] = df_info["Movie Info"].apply(lambda x: len(tokenizer.encode(x)))


,Title,Movie Info,Genre,num_tokens
0,Avatar,A paraplegic Marine dispatched to the moon Pan...,"['Action', 'Adventure', 'Fantasy', 'Sci-Fi']",31
1,Avengers: Endgame,After the devastating events of Avengers: Infi...,"['Action', 'Adventure', 'Drama', 'Sci-Fi']",43
2,Avatar: The Way of Water,Jake Sully lives with his newfound family form...,"['Action', 'Adventure', 'Drama', 'Sci-Fi']",49
3,Titanic,A seventeen-year-old aristocrat falls in love ...,"['Drama', 'Romance']",28
4,Star Wars: Episode VII - The Force Awakens,"As a new threat to the galaxy rises, Rey, a de...","['Action', 'Adventure', 'Sci-Fi']",44
...,...,...,...,...
995,Sweet Home Alabama,A young woman who has reinvented herself as a ...,"['Comedy', 'Romance']",33
996,Daddy's Home 2,Having finally gotten used to each other's exi...,['Comedy'],25
997,Hacksaw Ridge,World War II American Army Medic Desmond T. Do...,"['Biography', 'Drama', 'History', 'War']",44
998,Deja Vu,"After a ferry is bombed in New Orleans, an A.T...","['Action', 'Crime', 'Sci-Fi', 'Thriller']",40


In [ ]:
#  Extract smaller sample of the dataframe
smaller_df = df_info.sample(n=20)

In [ ]:
# Create Azure OpenAI client
client = AzureOpenAI(
  api_key = "your-azure-openai-api-key",  # Replace with your Azure OpenAI API key
  api_version = "your-azure-openai-api-version",  # Replace with your Azure OpenAI API version
  azure_endpoint = "your-azure-openai-endpoint",  # Replace with your Azure OpenAI endpoint
)

# Generate embedding using the deployed embedding model 
def generate_embeddings(text, model="text-embedding-ada-002"): # model = "deployment_name"
    return client.embeddings.create(input = [text], model=model).data[0].embedding

# Generate embeddings for the "Movie Info" column in the smaller_df dataframe
smaller_df['embeddings'] = smaller_df["Movie Info"].apply(lambda x : generate_embeddings (x, model = 'text-embedding-ada-002')) # model should be set to the deployment name you chose when you deployed the text-embedding-ada-002 (Version 2) model

In [34]:
smaller_df

,Title,Movie Info,Genre,num_tokens,embeddings
417,Home Alone 2: Lost in New York,One year after Kevin McCallister was left home...,"['Action', 'Drama', 'Family', 'Sport']",42,"[-0.0009052493842318654, -0.04048255831003189,..."
139,Interstellar,When Earth becomes uninhabitable in the future...,"['Adventure', 'Drama', 'Sci-Fi']",43,"[0.0002725477097555995, -0.04167971760034561, ..."
49,Harry Potter and the Sorcerer's Stone,An orphaned boy enrolls in a school of wizardr...,"['Adventure', 'Family', 'Fantasy']",34,"[0.007969120517373085, -0.03190186247229576, -..."
569,Sex and the City 2,"While wrestling with the pressures of life, lo...","['Comedy', 'Drama', 'Romance']",44,"[0.001377045875415206, -0.030631864443421364, ..."
883,Kindergarten Cop,A tough cop must pose as a kindergarten teache...,"['Action', 'Comedy', 'Crime']",31,"[0.00030547913047485054, 0.0006718905060552061..."
84,Guardians of the Galaxy Vol. 2,The Guardians struggle to keep together as a t...,"['Action', 'Adventure', 'Comedy', 'Sci-Fi']",34,"[0.016701050102710724, -0.030887963250279427, ..."
920,Wallace & Gromit: The Curse of the Were-Rabbit,"Wallace and his loyal dog, Gromit, set out to ...","['Adventure', 'Animation', 'Comedy', 'Family',...",35,"[0.01205082330852747, -0.029459016397595406, -..."
779,Knocked Up,"For fun-loving party animal Ben Stone, the las...","['Comedy', 'Romance']",39,"[-0.015663862228393555, -0.02305101603269577, ..."
11,Top Gun: Maverick,"After thirty years, Maverick is still pushing ...","['Action', 'Crime', 'Thriller']",49,"[-0.01213492825627327, -0.04286118969321251, -..."
608,Gran Torino,After a Hmong teenager tries to steal his priz...,['Drama'],36,"[-0.001315946807153523, -0.026475675404071808,..."


### Cosine similarity for vectors distance calculation

![cosine similarity](https://miro.medium.com/v2/resize:fit:720/format:webp/1*iV-kVEuRcz_Qpl2AJxiKMA.jpeg)

In [ ]:
# Function to calculate cosine similarity
def cosine_similarity_calculation(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

In [ ]:
# Function to convert user input into an embedding
def get_embedding(text, model="text-embedding-ada-002"): # model = "deployment_name"
    return client.embeddings.create(input = [text], model=model).data[0].embedding

# Function to search for documents based on user query
def search_docs(df, user_query, top_n=4, to_print=True):
    embedding = get_embedding(
        user_query,
        model="text-embedding-ada-002" # model should be set to the deployment name you chose when you deployed the text-embedding-ada-002 (Version 2) model
    )
    df["similarities"] = df["embeddings"].apply(lambda x: cosine_similarity_calculation(x, embedding))

    result = (
        df.sort_values("similarities", ascending=False)
        .head(top_n)
    )
    if to_print:
        display(result)
    return result

# Example usage of the search_docs function
result = search_docs(smaller_df, "something magical, unreal and classic", top_n=4)

,Title,Movie Info,Genre,num_tokens,embeddings,similarities
49,Harry Potter and the Sorcerer's Stone,An orphaned boy enrolls in a school of wizardr...,"['Adventure', 'Family', 'Fantasy']",34,"[0.007969120517373085, -0.03190186247229576, -...",0.805358
817,"Crouching Tiger, Hidden Dragon",A young Chinese warrior steals a sword from a ...,"['Action', 'Adventure', 'Drama', 'Fantasy', 'R...",32,"[0.0005399049841798842, -0.031106362119317055,...",0.793012
718,A.I. Artificial Intelligence,A highly advanced robotic boy longs to become ...,"['Drama', 'Sci-Fi']",24,"[-0.005350840277969837, -0.026822427287697792,...",0.773862
759,Better Days,While she copes the the pressures of her final...,"['Action', 'Adventure', 'Fantasy', 'Sci-Fi']",33,"[-0.013005590997636318, -0.007388947997242212,...",0.770694


In [50]:
result["Movie Info"][49]

'An orphaned boy enrolls in a school of wizardry, where he learns the truth about himself, his family and the terrible evil that haunts the magical world.'

What's next:
- Review your documents and identify strategies to split and vectorise them
- Review various vector store options such as Chroma, Pinecone, Qdrant, Azure AI Search, etc.
- How to enable RAG in LLM applications 